# Street View Insights (`SV_PANO` + Minimal 3D LiDAR): Address-to-House Localization & Feature Extraction

[Open in Colab](https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/address_to_house_triangulation_with_gmp_and_lidar_v2.ipynb) · [Open in Colab Enterprise](https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fsvi-gemini-improvements%2Fstreet_view_insights%2Fpanoramic%2Fnotebooks%2Faddress_to_house_triangulation_with_gmp_and_lidar_v2.ipynb) · [View on GitHub](https://github.com/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/address_to_house_triangulation_with_gmp_and_lidar_v2.ipynb)

**Expected runtime:** ~1–2 min per address (~$0.03–$0.06 in Gemini + API calls; BigQuery scan billed at $6.25/TB).

Given a **US street address**, this notebook combines **Google Maps Platform APIs** with **Street View Insights panoramic imagery (`SV_PANO`)** and an **on-demand, minimal 3D LiDAR fallback** to find the exact target building, extract multi-angle porch/facade imagery, triangulate its 3D entrance and building coordinates, and summarize delivery & property attributes:

| Stage | What happens | Services used |
| :--- | :--- | :--- |
| **1. Parcel & Curbside Prior** | Resolve building outline (`displayPolygon`), curbside `navigationPoints` (access street + side-of-road offset), multi-unit count (`subDestinations`), and approach side (`sideOfRoad: true`) | Geocoding API v4 (`SearchDestinations`), Routes API v2, Places API (New) fallback |
| **2. Near-Curb Frustum Selection** | Query nearby `SV_PANO` captures around the curbside arrival point, favor near-side curb passes, pick up to 6 diverse viewing angles, and project `20 MP` parcel crop windows | BigQuery (`pano_observations_latest`), fisheye camera geometry |
| **3. Target-Conditioned Zoom-Crop** | Crop each `3648 × 5472` frame around the projected parcel window for `2–3x` sharper door, porch, behind-tree wall, house-number OCR, and privacy-blur detection | Gemini (`gemini-4-argon`) on Vertex AI, Cloud Storage (in-memory) |
| **4. Multi-Ray Triangulation** | Map `0..1000` boxes losslessly back to full-frame pixels and solve RANSAC + Huber-IRLS ray intersection (`0` LiDAR bytes when `>= 3` car positions converge) | NumPy / Shapely |
| **5. Surgical 3D LiDAR Fallback** | **Triggered only when RGB `< 3` converging positions** (`~3%` of US addresses): stream `1–2` nearby `.laz` sweeps, peel foreground foliage/cars, and lock the 3D wall plane | BigQuery (`full_frame_lidar_metadata_latest`), `laspy` (in-memory) |

### Two-Tier Output Contract

* **Tier A (`TIER_A_RGB` or `TIER_A_LIDAR_ASSISTED`)**: Located from **3+ distinct SVI car positions** (`ray_rms_m <= 1.75 m`), or upgraded via **surgical 3D LiDAR wall-plane intersection** when foliage/corners limit RGB to 1–2 views.
* **Tier B (`TIER_B_RGB`)**: Seen from **1–2 car positions** where no nearby LiDAR sweep is available; retains all matched images, attributes, and a ray-to-building-outline coordinate estimate.
* **Explicit Street View Opt-Out Flag (`PRIVACY_BLURRED_IN_SVI`)**: Detects when a property owner has opted out of Street View imagery (rectangular Gaussian blur) instead of silently failing.

## 1. Before you start

**Enable these APIs** in your Google Cloud project:
* **BigQuery API**, **Vertex AI API**, and **Cloud Storage API** (for Street View Insights tables, imagery, and Gemini)
* **Geocoding API** (`geocoding-backend.googleapis.com`) and **Routes API** (`routes.googleapis.com`) — plus optional **Places API (New)** (`places.googleapis.com`) fallback

**Required IAM roles:**

| Role | Purpose |
| :--- | :--- |
| `roles/bigquery.jobUser` & `roles/bigquery.dataViewer` | Run parameterized queries against `imagery_insights___us` (and optional `imagery_insights_lidar___us`) |
| `roles/storage.objectViewer` | Stream Street View Insights JPEG frames and `.laz` sweeps in memory |
| `roles/aiplatform.user` | Call `gemini-4-argon` on Vertex AI |
| `roles/serviceusage.serviceUsageConsumer` | Attribute Geocoding v4, Routes v2, and Places v1 REST calls via `X-Goog-User-Project` |

**Terms of Service & Attribution:**
* Usage of Google Maps Platform APIs and Street View Insights is subject to the [Google Maps Platform Terms of Service](https://cloud.google.com/maps-platform/terms?utm_campaign=cloud_express_gmp) (including regional terms).
* Always display **"Imagery © Google"** and **"Map data © Google"** alongside rendered Street View frames and maps, stream imagery in memory without bulk offline caching or redistribution, and never use Google Maps content to train or fine-tune models.
* Note that `geocode/v4beta/geocode/destinations` is a Pre-GA (`v4beta`) endpoint provided "as-is" under the [Google Cloud Launch Stages](https://cloud.google.com/terms/launch-stages?utm_campaign=cloud_express_gmp).

In [ ]:
# @title Step 1a: Install packages
%pip install -q --upgrade google-genai google-cloud-bigquery db-dtypes shapely "laspy[lazrs]"

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# @title Step 1b: Imports & authentication
import concurrent.futures as cf
import dataclasses
from dataclasses import dataclass
import io
import json
import math
import os
import random
import re
import sys
import threading
import time
import urllib.parse
import warnings
from typing import Any, Mapping, Sequence

import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from google import genai
from google.genai import types
import laspy
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field
import shapely
from shapely.geometry import LineString, MultiPolygon, Point, Polygon, shape
from shapely.ops import nearest_points

warnings.filterwarnings("ignore", category=UserWarning)

if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()

print("Libraries imported and ready.")

## 2. Configure your project and target US address

* `PROJECT_ID`: Google Cloud project that bills BigQuery, Vertex AI, and Google Maps Platform calls.
* `SVI_PROJECT_ID` / `SVI_DATASET`: your subscribed Street View Insights panoramic dataset (`imagery_insights___us`).
* `SVI_LIDAR_DATASET` & `ENABLE_SURGICAL_LIDAR_FALLBACK`: optional US 3D LiDAR dataset (`imagery_insights_lidar___us`), invoked **only** when `< 3` RGB car positions converge (`0` `.laz` bytes fetched on ~97% of US addresses).

In [ ]:
# @title Step 2: Configuration
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
SVI_PROJECT_ID = ""  # @param {type:"string"}
SVI_DATASET = "imagery_insights___us"  # @param {type:"string"}
SVI_LIDAR_DATASET = "imagery_insights_lidar___us"  # @param {type:"string"}
VERTEX_LOCATION = "global"  # @param {type:"string"}
GEMINI_MODEL = "gemini-4-argon"  # @param {type:"string"}

TARGET_ADDRESS = "530 W Myrtle St, Lakeland, FL 33815"  # @param {type:"string"}
MAX_PANOS = 6  # @param {type:"integer"}
ENABLE_SURGICAL_LIDAR_FALLBACK = True  # @param {type:"boolean"}
MAX_QUERY_GB = 5.0  # @param {type:"number"}

GEMINI_INPUT_USD_PER_M = 2.00  # @param {type:"number"}
GEMINI_OUTPUT_USD_PER_M = 10.00  # @param {type:"number"}

if PROJECT_ID == "your-gcp-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
SVI_PROJECT_ID = SVI_PROJECT_ID or os.environ.get("SVI_PROJECT_ID", "") or PROJECT_ID
assert PROJECT_ID != "your-gcp-project-id", "Set PROJECT_ID to your Google Cloud project ID."

bq_client = bigquery.Client(project=PROJECT_ID)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_http = AuthorizedSession(creds)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)

print(f"Billing project: {PROJECT_ID} | SVI dataset: {SVI_PROJECT_ID}.{SVI_DATASET}")
print(f"Target address:  {TARGET_ADDRESS} | Surgical LiDAR fallback: {ENABLE_SURGICAL_LIDAR_FALLBACK}")

## 3. Stage 1: Resolve parcel outline, access street & side-of-road via Google Maps Platform

A basic rooftop geocode `(lat, lng)` fails on corner lots (picking a blank side wall on the wrong street), divided multi-lane roads (picking captures across oncoming traffic), and sub-premises (`POINT` units inside duplexes or apartment complexes).

Instead, `resolve_address_gmp()` combines two Google Maps Platform calls:
1. **Geocoding API v4 `SearchDestinations`** (`POST /v4beta/geocode/destinations`):
   * **`displayPolygon`**: 2D building footprint outline (resolved directly on `primary`, via `containingPlaces` when the address is a unit `POINT`, or via a `structureType: BUILDING` fallback).
   * **`navigationPoints`**: curbside arrival coordinate **offset from the road centerline toward the building's side of the street**, plus the true access street name (`displayName`).
   * **`subDestinations` & `entrances`**: multi-unit / apartment count and preferred entrance coordinates when available.
2. **Routes API v2 `computeRoutes`** (`POST /directions/v2:computeRoutes` with `sideOfRoad: true`):
   * Confirms the curb-side arrival point, vehicle approach bearing, and `RIGHT` vs. `LEFT` arrival maneuver.

From the building polygon and curbside `navigationPoint`, we extract the exact street-facing front wall `[front_left, front_mid, front_right]` and true inward building half-depth (`true_half_depth_m`).

In [ ]:
# @title Step 3: Parcel, curbside & side-of-road resolver (Geocoding v4 + Routes v2)
R_EARTH_M = 6_371_008.8
WGS84_A = 6_378_137.0
WGS84_E2 = 0.00669437999014
CAM_HEIGHT_ABOVE_GROUND_M = 2.5


def lla_to_en(lat: float, lng: float, ref_lat: float, ref_lng: float) -> tuple[float, float]:
    east = math.radians(lng - ref_lng) * R_EARTH_M * math.cos(math.radians(0.5 * (lat + ref_lat)))
    north = math.radians(lat - ref_lat) * R_EARTH_M
    return east, north


def en_to_lla(east: float, north: float, ref_lat: float, ref_lng: float) -> tuple[float, float]:
    lat = ref_lat + math.degrees(north / R_EARTH_M)
    lng = ref_lng + math.degrees(east / (R_EARTH_M * math.cos(math.radians(0.5 * (lat + ref_lat)))))
    return lat, lng


def distance_m(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
    e, n = lla_to_en(lat2, lng2, lat1, lng1)
    return math.hypot(e, n)


def ang_diff_deg(a: float, b: float) -> float:
    return (a - b + 180.0) % 360.0 - 180.0


def angle_between_rays_deg(az1: float, az2: float) -> float:
    d = abs(ang_diff_deg(az1, az2))
    return min(d, 180.0 - d)


def extract_street_number(address: str | None) -> str | None:
    if not address:
        return None
    m = re.match(r"^\s*(\d+(?:-\d+)?[A-Za-z]?)\b", address.strip())
    return m.group(1) if m else None


@dataclass
class ParcelPrior:
    query_address: str
    formatted_address: str
    place_id: str | None
    street_number: str | None
    route_name: str | None
    access_street_name: str | None
    ref_lat: float
    ref_lng: float
    nav_lat: float
    nav_lng: float
    side_of_road_lat: float
    side_of_road_lng: float
    approach_heading_deg: float | None
    arrival_side: str
    is_multi_unit: bool
    sub_destination_count: int
    polygon_source: str
    poly_en: Polygon | None
    front_left_en: np.ndarray
    front_mid_en: np.ndarray
    front_right_en: np.ndarray
    inward_normal_en: np.ndarray
    true_half_depth_m: float
    facade_width_m: float
    preferred_entrance_en: np.ndarray | None


def _compute_front_facade_from_polygon_and_nav(
    query_address: str,
    fmt_addr: str,
    place_id: str | None,
    street_num: str | None,
    route_name: str | None,
    access_street: str | None,
    ref_lat: float,
    ref_lng: float,
    nav_lat: float,
    nav_lng: float,
    sor_lat: float,
    sor_lng: float,
    approach_hdg: float | None,
    arrival_side: str,
    sub_cnt: int,
    poly_src: str,
    g_poly: Polygon | None,
    pref_ent_en: np.ndarray | None,
) -> ParcelPrior:
    nav_en = np.array(lla_to_en(nav_lat, nav_lng, ref_lat, ref_lng), dtype=np.float64)
    poly_en = Polygon([lla_to_en(y, x, ref_lat, ref_lng) for x, y in g_poly.exterior.coords]) if g_poly is not None else None

    if poly_en is not None and poly_en.area >= 12.0:
        coords = list(poly_en.exterior.coords)
        c_pt = np.array([poly_en.centroid.x, poly_en.centroid.y], dtype=np.float64)
        to_curb = nav_en - c_pt
        curb_dir = to_curb / max(float(np.linalg.norm(to_curb)), 1e-3)
        best_edge = None
        best_score = -1e9
        nav_geom = Point(float(nav_en[0]), float(nav_en[1]))
        for idx in range(len(coords) - 1):
            p0 = np.array(coords[idx], dtype=np.float64)
            p1 = np.array(coords[idx + 1], dtype=np.float64)
            seg_vec = p1 - p0
            seg_len = float(np.linalg.norm(seg_vec))
            if seg_len < 2.2:
                continue
            mid = 0.5 * (p0 + p1)
            n1 = np.array([-seg_vec[1], seg_vec[0]], dtype=np.float64) / seg_len
            out_n = -n1 if poly_en.contains(Point(float(mid[0] + 0.25 * n1[0]), float(mid[1] + 0.25 * n1[1]))) else n1
            dist_nav = float(LineString([p0, p1]).distance(nav_geom))
            to_nav_u = (nav_en - mid) / max(float(np.linalg.norm(nav_en - mid)), 1e-3)
            score = 14.0 * float(np.dot(out_n, to_nav_u)) - 1.15 * dist_nav + 0.35 * min(seg_len, 16.0)
            if score > best_score:
                best_score = score
                best_edge = (p0, p1, mid, -out_n, seg_len)

        if best_edge is not None:
            p0, p1, mid, in_n, seg_len = best_edge
            look_u = (mid - nav_en) / max(float(np.linalg.norm(mid - nav_en)), 1e-3)
            right_u = np.array([look_u[1], -look_u[0]], dtype=np.float64)
            f_left, f_right = (p0, p1) if float(np.dot(p0 - mid, right_u)) <= float(np.dot(p1 - mid, right_u)) else (p1, p0)
            all_pts = np.array(coords[:-1], dtype=np.float64)
            front_pts = all_pts[((all_pts - mid) @ in_n) <= 3.2]
            if len(front_pts) >= 2:
                lat_proj = (front_pts - mid) @ right_u
                f_left = mid + float(lat_proj.min()) * right_u
                f_right = mid + float(lat_proj.max()) * right_u
                mid = 0.5 * (f_left + f_right)
                seg_len = float(np.linalg.norm(f_right - f_left))
            half_depth = max(2.5, min(14.0, float(np.dot(c_pt - mid, in_n))))
            inward_n, front_left, front_mid, front_right, facade_w = in_n, f_left, mid, f_right, max(seg_len, 4.0)
        else:
            inward_n = -curb_dir
            front_mid = c_pt + 5.0 * curb_dir
            right_u = np.array([-curb_dir[1], curb_dir[0]], dtype=np.float64)
            front_left, front_right, half_depth, facade_w = front_mid - 5.0 * right_u, front_mid + 5.0 * right_u, 5.5, 10.0
    else:
        nrm = float(np.linalg.norm(nav_en))
        curb_dir = nav_en / nrm if nrm > 1.0 else np.array([0.0, 1.0], dtype=np.float64)
        inward_n = -curb_dir
        front_mid = min(5.5, 0.35 * nrm) * curb_dir if nrm > 3.0 else np.zeros(2, dtype=np.float64)
        right_u = np.array([-curb_dir[1], curb_dir[0]], dtype=np.float64)
        front_left, front_right, half_depth, facade_w = front_mid - 5.0 * right_u, front_mid + 5.0 * right_u, 5.8, 10.0

    return ParcelPrior(
        query_address=query_address,
        formatted_address=fmt_addr,
        place_id=place_id,
        street_number=street_num,
        route_name=route_name,
        access_street_name=access_street,
        ref_lat=ref_lat,
        ref_lng=ref_lng,
        nav_lat=nav_lat,
        nav_lng=nav_lng,
        side_of_road_lat=sor_lat,
        side_of_road_lng=sor_lng,
        approach_heading_deg=approach_hdg,
        arrival_side=arrival_side,
        is_multi_unit=(sub_cnt > 0),
        sub_destination_count=sub_cnt,
        polygon_source=poly_src,
        poly_en=poly_en,
        front_left_en=front_left,
        front_mid_en=front_mid,
        front_right_en=front_right,
        inward_normal_en=inward_n,
        true_half_depth_m=half_depth,
        facade_width_m=facade_w,
        preferred_entrance_en=pref_ent_en,
    )


def resolve_address_gmp(address: str) -> ParcelPrior:
    """Resolve building outline, curbside navigationPoint, multi-unit status, and side-of-road approach."""
    dest_mask = ",".join([
        "destinations.primary",
        "destinations.containingPlaces",
        "destinations.subDestinations.formattedAddress",
        "destinations.entrances",
        "destinations.navigationPoints",
    ])
    headers = {"X-Goog-User-Project": PROJECT_ID, "X-Goog-FieldMask": dest_mask}
    r = authed_http.post(
        "https://geocode.googleapis.com/v4beta/geocode/destinations",
        headers=headers,
        json={"addressQuery": {"addressQuery": address}},
        timeout=20,
    )
    dests = (r.json().get("destinations") or []) if r.status_code == 200 else []
    d0 = dests[0] if dests else {}
    prim = d0.get("primary") or {}

    # Fallback to Places API (New) Text Search if SearchDestinations returned empty
    if not prim:
        rp = authed_http.post(
            "https://places.googleapis.com/v1/places:searchText",
            headers={"X-Goog-User-Project": PROJECT_ID, "X-Goog-FieldMask": "places.id,places.formattedAddress,places.location"},
            json={"textQuery": address},
            timeout=20,
        )
        rp.raise_for_status()
        places = rp.json().get("places") or []
        if not places:
            raise ValueError(f"Could not resolve address: {address!r}")
        prim = {"placeId": places[0].get("id"), "formattedAddress": places[0].get("formattedAddress"), "location": places[0].get("location")}

    fmt_addr = prim.get("formattedAddress") or address
    place_id = prim.get("placeId")
    street_num = extract_street_number(address) or extract_street_number(fmt_addr)
    route_name = None
    for comp in prim.get("addressComponents") or []:
        if "route" in (comp.get("types") or []):
            route_name = comp.get("shortText") or comp.get("longText")
        if not street_num and "street_number" in (comp.get("types") or []):
            street_num = comp.get("shortText") or comp.get("longText")

    poly_geojson = prim.get("displayPolygon")
    poly_src = "primary" if poly_geojson else "none"
    if not poly_geojson:
        for cp in d0.get("containingPlaces") or []:
            if cp.get("displayPolygon") and cp.get("structureType") == "BUILDING":
                poly_geojson = cp["displayPolygon"]
                poly_src = "containingPlaces"
                break
    if not poly_geojson and prim.get("location"):
        r2 = authed_http.post(
            "https://geocode.googleapis.com/v4beta/geocode/destinations",
            headers=headers,
            json={"locationQuery": {"location": prim["location"], "placeFilter": {"structureType": "BUILDING"}}},
            timeout=20,
        )
        if r2.status_code == 200:
            for cd in r2.json().get("destinations") or []:
                if (cd.get("primary") or {}).get("displayPolygon"):
                    poly_geojson = cd["primary"]["displayPolygon"]
                    poly_src = "locationQuery_BUILDING"
                    break

    g_poly: Polygon | None = None
    if poly_geojson:
        shp = shape(poly_geojson)
        if isinstance(shp, MultiPolygon):
            shp = max(shp.geoms, key=lambda g: g.area)
        if isinstance(shp, Polygon):
            g_poly = shp

    if g_poly is not None:
        ref_lat, ref_lng = float(g_poly.centroid.y), float(g_poly.centroid.x)
    else:
        loc = prim.get("location") or {}
        ref_lat, ref_lng = float(loc["latitude"]), float(loc["longitude"])

    nav_pts = d0.get("navigationPoints") or []
    if nav_pts:
        nloc = nav_pts[0].get("location") or {}
        nav_lat, nav_lng = float(nloc.get("latitude", ref_lat)), float(nloc.get("longitude", ref_lng))
        access_street = (nav_pts[0].get("displayName") or {}).get("text") or route_name
    else:
        nav_lat, nav_lng = ref_lat, ref_lng
        access_street = route_name

    # Routes API v2 with sideOfRoad=True for arrival bearing and curb-side verification
    sor_lat, sor_lng = nav_lat, nav_lng
    approach_hdg = None
    arrival_side = "UNKNOWN"
    rt_mask = "routes.legs.endLocation,routes.legs.steps.startLocation,routes.legs.steps.endLocation,routes.legs.steps.navigationInstruction"
    dest_wp = {"placeId": place_id, "sideOfRoad": True} if place_id else {"address": fmt_addr, "sideOfRoad": True}
    rr = authed_http.post(
        "https://routes.googleapis.com/directions/v2:computeRoutes",
        headers={"X-Goog-User-Project": PROJECT_ID, "X-Goog-FieldMask": rt_mask},
        json={"origin": {"location": {"latLng": {"latitude": nav_lat + 0.0009, "longitude": nav_lng + 0.0009}}}, "destination": dest_wp, "travelMode": "DRIVE"},
        timeout=20,
    )
    if rr.status_code == 200:
        legs = ((rr.json().get("routes") or [{}])[0]).get("legs") or []
        if legs:
            end_ll = (legs[0].get("endLocation") or {}).get("latLng") or {}
            if "latitude" in end_ll:
                sor_lat, sor_lng = float(end_ll["latitude"]), float(end_ll["longitude"])
            steps = legs[0].get("steps") or []
            if steps:
                instr = ((steps[-1].get("navigationInstruction") or {}).get("instructions") or "").lower()
                arrival_side = "RIGHT" if "on the right" in instr else ("LEFT" if "on the left" in instr else "UNKNOWN")
                s_ll = (steps[-1].get("startLocation") or {}).get("latLng") or {}
                e_ll = (steps[-1].get("endLocation") or {}).get("latLng") or {}
                if "latitude" in s_ll and "latitude" in e_ll:
                    de, dn = lla_to_en(float(e_ll["latitude"]), float(e_ll["longitude"]), float(s_ll["latitude"]), float(s_ll["longitude"]))
                    if math.hypot(de, dn) >= 2.0:
                        approach_hdg = math.degrees(math.atan2(de, dn)) % 360.0

    pref_ent_en = None
    ents = d0.get("entrances") or []
    if ents:
        chosen_e = next((e for e in ents if "PREFERRED" in (e.get("tags") or [])), ents[0])
        eloc = chosen_e.get("location") or {}
        if "latitude" in eloc:
            pref_ent_en = np.array(lla_to_en(float(eloc["latitude"]), float(eloc["longitude"]), ref_lat, ref_lng), dtype=np.float64)

    sub_cnt = len(d0.get("subDestinations") or [])
    return _compute_front_facade_from_polygon_and_nav(
        address, fmt_addr, place_id, street_num, route_name, access_street,
        ref_lat, ref_lng, nav_lat, nav_lng, sor_lat, sor_lng, approach_hdg,
        arrival_side, sub_cnt, poly_src, g_poly, pref_ent_en,
    )


prior = resolve_address_gmp(TARGET_ADDRESS)
print(f"Resolved: {prior.formatted_address} (place_id={prior.place_id})")
print(f"  Building outline source: {prior.polygon_source} | width={prior.facade_width_m:.1f} m | half_depth={prior.true_half_depth_m:.1f} m")
print(f"  Access street: {prior.access_street_name} | Curbside navPoint: ({prior.nav_lat:.6f}, {prior.nav_lng:.6f}) | Arrival side: {prior.arrival_side}")
print(f"  Multi-unit complex: {prior.is_multi_unit} (sub-destinations={prior.sub_destination_count})")

## 4. Stage 2: Select near-curb, front-facing panoramic views & project 20 MP crop windows

Each Street View Insights panoramic rosette (`SV_PANO`) contains 6 horizontal portrait fisheye frames (`3648 × 5472` px, `cam_k = 0..5`, spaced `60°` apart) plus an upward zenith camera (`_6`). Note that `pano_id` can be `NULL` on ~74% of US `SV_PANO` rows, so we derive the canonical capture key directly from `observation_id` (`o1:<stem>_<k>:5001ee`).

To avoid sending 50–200 corridor frames to Gemini:
1. We query `pano_observations_latest` within `58 m` of the curbside `navigationPoint` (or building center on corner lots / deep setbacks), running a dry-run byte check first.
2. We project the 3D front wall `[front_left, front_mid, front_right]` into each candidate camera's pixel coordinates (`enu_offset_to_pixel`).
3. We segment captures into single car passes (`dt <= 60 s`, `|d_alt| <= 2.2 m`), prioritize **near-curb right-hand passes** where the camera looks nearly dead-center (`cx = 1835.6`) at the front wall, and pick up to `MAX_PANOS = 6` viewpoints with wide angular parallax spread (`~34x` fewer frames sent to Gemini).

In [ ]:
# @title Step 4: Fisheye lens model & BigQuery frustum selector
@dataclass(frozen=True)
class FisheyeLens:
    width: int = 3648
    height: int = 5472
    f_px: float = 2847.13  # approximate equidistant focal length estimated from imagery (px/rad)
    cx: float = 1835.6     # approximate optical center (px)
    cy: float = 2736.0
    k1: float = -0.028     # 1-term radial distortion


LENS = FisheyeLens()


def camera_to_enu_rotation(heading_deg: float, pitch_deg: float, roll_deg: float) -> np.ndarray:
    """3x3 matrix whose columns are camera [right, down, forward] axes in local East-North-Up."""
    h, p, r = map(math.radians, (heading_deg, pitch_deg, roll_deg))
    fwd = np.array([math.sin(h) * math.cos(p), math.cos(h) * math.cos(p), math.sin(p)], dtype=np.float64)
    right0 = np.array([math.cos(h), -math.sin(h), 0.0], dtype=np.float64)
    down0 = np.cross(fwd, right0)
    right = math.cos(r) * right0 + math.sin(r) * down0
    down = -math.sin(r) * right0 + math.cos(r) * down0
    return np.stack([right, down, fwd], axis=1)


def pixel_to_camera_ray(lens: FisheyeLens, u: float, v: float) -> np.ndarray:
    mx, my = (u - lens.cx) / lens.f_px, (v - lens.cy) / lens.f_px
    theta_d = math.hypot(mx, my)
    theta = theta_d
    for _ in range(20):
        f = theta * (1.0 + lens.k1 * theta * theta) - theta_d
        theta -= f / (1.0 + 3.0 * lens.k1 * theta * theta)
    s = math.sin(theta) / theta_d if theta_d > 1e-12 else 1.0
    return np.array([mx * s, my * s, math.cos(theta)], dtype=np.float64)


def pixel_to_bearing(
    lens: FisheyeLens, heading: float, pitch: float, roll: float, u: float, v: float
) -> tuple[float, float]:
    d = camera_to_enu_rotation(heading, pitch, roll) @ pixel_to_camera_ray(lens, u, v)
    az = math.degrees(math.atan2(d[0], d[1])) % 360.0
    el = math.degrees(math.asin(max(-1.0, min(1.0, float(d[2])))))
    return az, el


def enu_offset_to_pixel(
    lens: FisheyeLens, heading: float, pitch: float, roll: float, d_east: float, d_north: float, d_up: float
) -> tuple[float, float, float] | None:
    """Exact inverse of `pixel_to_bearing`: projects camera-relative ENU `[dE, dN, dU]` to `(u_px, v_px, fwd_m)`."""
    v_cam = camera_to_enu_rotation(heading, pitch, roll).T @ np.array([d_east, d_north, d_up], dtype=np.float64)
    if v_cam[2] <= 0.25:
        return None
    norm = float(np.linalg.norm(v_cam))
    theta = math.acos(max(-1.0, min(1.0, float(v_cam[2] / norm))))
    if theta > math.radians(62.0):
        return None
    theta_d = theta * (1.0 + lens.k1 * theta * theta)
    r_xy = math.hypot(float(v_cam[0]), float(v_cam[1]))
    scale = lens.f_px * (theta_d / r_xy) if r_xy > 1e-12 else lens.f_px
    return lens.cx + float(v_cam[0]) * scale, lens.cy + float(v_cam[1]) * scale, float(v_cam[2])


def query_nearby_pano_frames(prior: ParcelPrior, radius_m: float = 58.0, verbose: bool = True) -> pd.DataFrame:
    sql = f"""
    SELECT
      observation_id,
      COALESCE(NULLIF(pano_id, ''), REGEXP_EXTRACT(observation_id, r'^o1:(.+)_[0-6]:5001ee$')) AS pano_id,
      CAST(REGEXP_EXTRACT(observation_id, r'_([0-6]):5001ee$') AS INT64) AS cam_k,
      capture_time,
      capture_location.latitude AS capture_lat,
      capture_location.longitude AS capture_lng,
      camera_pose.latitude AS cam_lat,
      camera_pose.longitude AS cam_lng,
      camera_pose.altitude AS cam_alt,
      camera_pose.heading AS heading,
      camera_pose.pitch AS pitch,
      camera_pose.roll AS roll,
      gcs_uri
    FROM `{SVI_PROJECT_ID}.{SVI_DATASET}.pano_observations_latest`
    WHERE (
      ST_DWITHIN(ST_GEOGPOINT(capture_location.longitude, capture_location.latitude), ST_GEOGPOINT(@nav_lng, @nav_lat), @radius_m)
      OR ST_DWITHIN(ST_GEOGPOINT(capture_location.longitude, capture_location.latitude), ST_GEOGPOINT(@ref_lng, @ref_lat), @radius_m)
    )
      AND REGEXP_CONTAINS(observation_id, r'_[0-5]:5001ee$')
    """
    params = [
        bigquery.ScalarQueryParameter("nav_lng", "FLOAT64", prior.nav_lng),
        bigquery.ScalarQueryParameter("nav_lat", "FLOAT64", prior.nav_lat),
        bigquery.ScalarQueryParameter("ref_lng", "FLOAT64", prior.ref_lng),
        bigquery.ScalarQueryParameter("ref_lat", "FLOAT64", prior.ref_lat),
        bigquery.ScalarQueryParameter("radius_m", "FLOAT64", radius_m),
    ]
    dry_cfg = bigquery.QueryJobConfig(query_parameters=params, dry_run=True, use_query_cache=False)
    est_gb = bq_client.query(sql, job_config=dry_cfg).total_bytes_processed / 1e9
    assert est_gb <= MAX_QUERY_GB, f"Estimated scan {est_gb:.2f} GB exceeds MAX_QUERY_GB={MAX_QUERY_GB}"
    if verbose:
        print(f"BigQuery dry-run: {est_gb:.2f} GB estimated (cap {MAX_QUERY_GB:.1f} GB)")
    run_cfg = bigquery.QueryJobConfig(query_parameters=params, maximum_bytes_billed=int(MAX_QUERY_GB * 1e9))
    df = bq_client.query(sql, job_config=run_cfg).to_dataframe()
    if not df.empty:
        df["capture_time"] = pd.to_datetime(df["capture_time"], utc=True)
    return df


def compute_projected_crop_window(
    prior: ParcelPrior, cam_lat: float, cam_lng: float, heading: float, pitch: float, roll: float
) -> dict[str, Any] | None:
    cam_e, cam_n = lla_to_en(cam_lat, cam_lng, prior.ref_lat, prior.ref_lng)
    cam_pt = np.array([cam_e, cam_n], dtype=np.float64)
    nav_e, nav_n = lla_to_en(prior.nav_lat, prior.nav_lng, prior.ref_lat, prior.ref_lng)
    setback_m = float(np.linalg.norm(np.array([nav_e, nav_n]) - prior.front_mid_en))
    max_front_dist = max(48.0, min(72.0, setback_m + 24.0))

    to_cam = cam_pt - prior.front_mid_en
    dist_front = float(np.linalg.norm(to_cam))
    if not (3.2 <= dist_front <= max_front_dist):
        return None
    outward_n = -prior.inward_normal_en
    cos_inc = float(np.dot(to_cam / dist_front, outward_n))
    if cos_inc < -0.15:
        return None

    mid_proj = enu_offset_to_pixel(LENS, heading, pitch, roll, float(prior.front_mid_en[0] - cam_e), float(prior.front_mid_en[1] - cam_n), -0.4)
    if mid_proj is None or not (120.0 <= mid_proj[0] <= LENS.width - 120.0):
        return None
    u_mid, v_mid, _ = mid_proj

    pts_2d = [prior.front_left_en, prior.front_mid_en, prior.front_right_en]
    if prior.poly_en is not None:
        pts_2d.extend(np.array(c, dtype=np.float64) for c in list(prior.poly_en.exterior.coords)[:-1])
    us, vs = [], []
    for pt in pts_2d:
        de, dn = float(pt[0] - cam_e), float(pt[1] - cam_n)
        for du in (-CAM_HEIGHT_ABOVE_GROUND_M - 0.6, 0.5, 6.2):
            pr = enu_offset_to_pixel(LENS, heading, pitch, roll, de, dn, du)
            if pr is not None:
                us.append(pr[0])
                vs.append(pr[1])
    if len(us) < 2:
        return None

    raw_x0, raw_x1 = max(0.0, min(us)), min(float(LENS.width), max(us))
    raw_y0, raw_y1 = max(0.0, min(vs)), min(float(LENS.height), max(vs))
    pad_x = max(0.48 * max(raw_x1 - raw_x0, 260.0), 420.0)
    pad_y_top = max(0.40 * max(raw_y1 - raw_y0, 480.0), 360.0)
    pad_y_bot = max(0.32 * max(raw_y1 - raw_y0, 480.0), 280.0)

    cx0 = int(max(0, math.floor(min(raw_x0, u_mid - 250.0) - pad_x)))
    cx1 = int(min(LENS.width, math.ceil(max(raw_x1, u_mid + 250.0) + pad_x)))
    cy0 = int(max(0, math.floor(min(raw_y0, v_mid - 350.0) - pad_y_top)))
    cy1 = int(min(LENS.height, math.ceil(max(raw_y1, v_mid + 350.0) + pad_y_bot)))
    if cx1 - cx0 < 1280:
        h_need = (1280 - (cx1 - cx0)) // 2 + 1
        cx0, cx1 = max(0, cx0 - h_need), min(LENS.width, cx1 + h_need)
    if cy1 - cy0 < 1440:
        v_need = (1440 - (cy1 - cy0)) // 2 + 1
        cy0, cy1 = max(0, cy0 - v_need), min(LENS.height, cy1 + v_need)

    cw = max(cx1 - cx0, 1)
    az_from_house = math.degrees(math.atan2(to_cam[0], to_cam[1])) % 360.0
    out_az = math.degrees(math.atan2(outward_n[0], outward_n[1])) % 360.0
    return {
        "dist_front_m": round(dist_front, 2),
        "dist_nav_m": round(math.hypot(cam_e - nav_e, cam_n - nav_n), 2),
        "cos_incidence": round(cos_inc, 3),
        "oblique_deg": round(ang_diff_deg(az_from_house, out_az), 1),
        "u_mid_px": round(u_mid, 1),
        "v_mid_px": round(v_mid, 1),
        "crop_box": (cx0, cy0, cx1, cy1),
        "exp_x0_1000": int(np.clip(round(1000.0 * (raw_x0 - cx0) / cw), 0, 1000)),
        "exp_x1_1000": int(np.clip(round(1000.0 * (raw_x1 - cx0) / cw), 0, 1000)),
        "exp_u_mid_1000": int(np.clip(round(1000.0 * (u_mid - cx0) / cw), 0, 1000)),
    }


def select_optimal_house_frames(
    prior: ParcelPrior, local_df: pd.DataFrame, max_panos: int = 6, allow_second_drive: bool = True
) -> tuple[pd.DataFrame, dict[str, Any]]:
    if local_df.empty:
        return local_df.copy(), {"pool_frames": 0, "pool_panos": 0, "selected_panos": 0, "selected_frames": 0, "drives_used": 0, "frame_reduction_x": 0.0}

    pano_meta = (
        local_df.groupby("pano_id")
        .agg(capture_time=("capture_time", "min"), cap_lat=("capture_lat", "first"), cap_lng=("capture_lng", "first"), cam_alt=("cam_alt", "median"))
        .reset_index().sort_values("capture_time").reset_index(drop=True)
    )
    drive_ids = [0]
    for idx in range(1, len(pano_meta)):
        dt = (pano_meta.loc[idx, "capture_time"] - pano_meta.loc[idx - 1, "capture_time"]).total_seconds()
        dalt = abs(float(pano_meta.loc[idx, "cam_alt"] or 0.0) - float(pano_meta.loc[idx - 1, "cam_alt"] or 0.0))
        dpos = distance_m(float(pano_meta.loc[idx, "cap_lat"]), float(pano_meta.loc[idx, "cap_lng"]), float(pano_meta.loc[idx - 1, "cap_lat"]), float(pano_meta.loc[idx - 1, "cap_lng"]))
        drive_ids.append(drive_ids[-1] + (1 if (dt > 60.0 or dalt > 2.2 or dpos > 45.0) else 0))
    pano_meta["drive_id"] = drive_ids
    drive_map = dict(zip(pano_meta["pano_id"], pano_meta["drive_id"]))

    drive_headings: dict[int, float | None] = {}
    for did, grp in pano_meta.groupby("drive_id"):
        if len(grp) >= 2:
            de, dn = lla_to_en(float(grp.iloc[-1]["cap_lat"]), float(grp.iloc[-1]["cap_lng"]), float(grp.iloc[0]["cap_lat"]), float(grp.iloc[0]["cap_lng"]))
            drive_headings[int(did)] = math.degrees(math.atan2(de, dn)) % 360.0 if math.hypot(de, dn) >= 3.0 else None
        else:
            drive_headings[int(did)] = None

    def _collect(active_prior: ParcelPrior) -> list[dict[str, Any]]:
        rows_out = []
        for r in local_df.itertuples(index=False):
            proj = compute_projected_crop_window(active_prior, float(r.cam_lat), float(r.cam_lng), float(r.heading), float(r.pitch), float(r.roll))
            if proj is None:
                continue
            did = int(drive_map[r.pano_id])
            d_hdg = drive_headings.get(did)
            same_side_bonus = 0.0
            if d_hdg is not None:
                ce, cn = lla_to_en(float(r.cam_lat), float(r.cam_lng), active_prior.ref_lat, active_prior.ref_lng)
                to_house = active_prior.front_mid_en - np.array([ce, cn])
                if 20.0 <= ang_diff_deg(math.degrees(math.atan2(to_house[0], to_house[1])) % 360.0, d_hdg) <= 160.0:
                    same_side_bonus = 2.5
            dist_f = proj["dist_front_m"]
            dist_pen = 0.0 if 6.5 <= dist_f <= 22.0 else (0.22 * (dist_f - 22.0) if dist_f > 22.0 else 0.5 * (6.5 - dist_f))
            centrality = max(0.0, 1.0 - abs(float(proj["u_mid_px"]) - LENS.cx) / (0.5 * LENS.width))
            score = 4.5 * proj["cos_incidence"] + 5.5 * centrality + same_side_bonus - dist_pen - 0.08 * max(0.0, proj["dist_nav_m"] - 18.0)
            rows_out.append({**r._asdict(), "drive_id": did, "frame_score": round(score, 3), **proj})
        return rows_out

    cand_rows = _collect(prior)
    by_dr = {}
    for x in cand_rows:
        by_dr.setdefault(x["drive_id"], set()).add(x["pano_id"])
    best_dr_cnt = max((len(s) for s in by_dr.values()), default=0)

    # Corner-building fallback: if navigationPoint sits on an un-captured side street, re-orient toward the captured street
    if best_dr_cnt < 3 and len(pano_meta) >= 3 and prior.poly_en is not None:
        g_poly_wgs = Polygon([en_to_lla(x, y, prior.ref_lat, prior.ref_lng)[::-1] for x, y in prior.poly_en.exterior.coords])
        for _, d_grp in pano_meta.groupby("drive_id"):
            p_en = np.array([lla_to_en(float(r.cap_lat), float(r.cap_lng), prior.ref_lat, prior.ref_lng) for r in d_grp.itertuples()])
            ni = int(np.argmin(np.hypot(p_en[:, 0], p_en[:, 1])))
            alt_p = _compute_front_facade_from_polygon_and_nav(
                prior.query_address, prior.formatted_address, prior.place_id, prior.street_number,
                prior.route_name, prior.route_name or prior.access_street_name, prior.ref_lat, prior.ref_lng,
                float(d_grp.iloc[ni]["cap_lat"]), float(d_grp.iloc[ni]["cap_lng"]),
                prior.side_of_road_lat, prior.side_of_road_lng, prior.approach_heading_deg,
                prior.arrival_side, prior.sub_destination_count, prior.polygon_source, g_poly_wgs, prior.preferred_entrance_en,
            )
            alt_rows = _collect(alt_p)
            alt_dr = {}
            for x in alt_rows:
                alt_dr.setdefault(x["drive_id"], set()).add(x["pano_id"])
            cnt = max((len(s) for s in alt_dr.values()), default=0)
            if cnt > best_dr_cnt:
                best_dr_cnt, cand_rows = cnt, alt_rows
                prior.front_left_en, prior.front_mid_en, prior.front_right_en = alt_p.front_left_en, alt_p.front_mid_en, alt_p.front_right_en
                prior.inward_normal_en, prior.true_half_depth_m, prior.facade_width_m = alt_p.inward_normal_en, alt_p.true_half_depth_m, alt_p.facade_width_m

    if not cand_rows:
        return local_df.iloc[0:0].copy(), {"pool_frames": int(len(local_df)), "pool_panos": int(local_df["pano_id"].nunique()), "selected_panos": 0, "selected_frames": 0, "drives_used": 0, "frame_reduction_x": 0.0}

    cdf = pd.DataFrame(cand_rows)
    best_per_pano = cdf.sort_values(["pano_id", "frame_score"], ascending=[True, False]).groupby("pano_id", as_index=False).first()
    drive_rank = (
        best_per_pano.groupby("drive_id")
        .agg(n_panos=("pano_id", "size"), n_good=("cos_incidence", lambda s: int((s >= 0.42).sum())), mean_score=("frame_score", "mean"), min_nav=("dist_nav_m", "min"), latest=("capture_time", "max"))
        .reset_index()
    )
    drive_rank["utility"] = 2.6 * np.minimum(drive_rank["n_good"], 5) + 1.2 * np.minimum(drive_rank["n_panos"], 6) + drive_rank["mean_score"] - 0.12 * np.maximum(0.0, drive_rank["min_nav"] - 14.0) + 0.35 * ((drive_rank["latest"].dt.year - 2018).clip(0, 10))
    drive_rank = drive_rank.sort_values("utility", ascending=False).reset_index(drop=True)

    def _pick_diverse(pool_df: pd.DataFrame, k_target: int, angles_so_far: list[float]) -> list[str]:
        chosen_ids, angles = [], list(angles_so_far)
        rem = pool_df.sort_values("frame_score", ascending=False).copy()
        while len(chosen_ids) < k_target and not rem.empty:
            if angles:
                ang_bonus = [min(abs(float(r.oblique_deg) - a) for a in angles) * 0.14 for r in rem.itertuples()]
                rem = rem.assign(_tot=rem["frame_score"] + ang_bonus).sort_values("_tot", ascending=False)
            top = rem.iloc[0]
            chosen_ids.append(str(top["pano_id"]))
            angles.append(float(top["oblique_deg"]))
            rem = rem[rem["pano_id"] != top["pano_id"]]
        return chosen_ids

    chosen_ids = _pick_diverse(best_per_pano[best_per_pano["drive_id"] == int(drive_rank.iloc[0]["drive_id"])], max_panos, [])
    drives_used = 1
    chosen_angles = best_per_pano.set_index("pano_id").loc[chosen_ids, "oblique_deg"].tolist()
    if allow_second_drive and (len(chosen_ids) < 4 or (max(chosen_angles) - min(chosen_angles) if len(chosen_angles) >= 2 else 0.0) < 14.0) and len(drive_rank) > 1:
        for idx_d in range(1, len(drive_rank)):
            sec_pool = best_per_pano[(best_per_pano["drive_id"] == int(drive_rank.iloc[idx_d]["drive_id"])) & (best_per_pano["cos_incidence"] >= 0.25)]
            if sec_pool.empty:
                continue
            extra = _pick_diverse(sec_pool, min(max(5 - len(chosen_ids), 2), max_panos - len(chosen_ids) + 1), chosen_angles)
            if extra:
                chosen_ids.extend(extra)
                chosen_angles = best_per_pano.set_index("pano_id").loc[chosen_ids, "oblique_deg"].tolist()
                drives_used += 1
            if len(chosen_ids) >= 5 and (max(chosen_angles) - min(chosen_angles)) >= 16.0:
                break

    selected = best_per_pano[best_per_pano["pano_id"].isin(chosen_ids)].sort_values(["capture_time", "cam_k"]).reset_index(drop=True)
    stats = {
        "pool_frames": int(len(local_df)),
        "pool_panos": int(local_df["pano_id"].nunique()),
        "selected_panos": int(selected["pano_id"].nunique()),
        "selected_frames": int(len(selected)),
        "drives_used": drives_used,
        "angular_span_deg": round(float(max(chosen_angles) - min(chosen_angles)) if len(chosen_angles) >= 2 else 0.0, 1),
        "frame_reduction_x": round(float(len(local_df)) / max(float(len(selected)), 1.0), 1),
    }
    return selected, stats


local_frames_df = query_nearby_pano_frames(prior)
selected_frames_df, frustum_stats = select_optimal_house_frames(prior, local_frames_df, max_panos=MAX_PANOS)
print(
    f"Frustum selection: {frustum_stats['pool_frames']} nearby frames ({frustum_stats['pool_panos']} rosettes) "
    f"-> {frustum_stats['selected_frames']} target-facing frames ({frustum_stats['frame_reduction_x']}x reduction, "
    f"angular span {frustum_stats['angular_span_deg']} deg across {frustum_stats['drives_used']} drive)"
)
selected_frames_df[["observation_id", "cam_k", "dist_front_m", "oblique_deg", "crop_box"]]

## 5. Stage 3: Target-conditioned 20 MP zoom-crop detection with Gemini

In a full `3648 × 5472` (`20 MP`) wide-angle frame, a house `20 m` away spans only ~20–25% of the width alongside neighboring houses, parked cars, and street trees.

By cropping around the projected parcel window `(cx0, cy0, cx1, cy1)` before calling `gemini-4-argon`:
* Gemini receives **`2–3x` higher effective detail** on front doors, recessed porches, house-number digits (`visible_house_number`), and walls partially behind street trees.
* Every normalized `0..1000` box maps **losslessly** back to full `3648 × 5472` sensor pixels via `cx0 + (x / 1000) * (cx1 - cx0)`.
* If an entire property is covered by a Street View owner-requested privacy blur box, Gemini returns `occlusion_level = "PRIVACY_BLURRED"` so downstream workflows receive an explicit explanation.

In [ ]:
# @title Step 5: Gemini schema, crop helper & parallel detection
class TargetHouseCropDetection(BaseModel):
    target_building_visible: bool = Field(
        description="True if the target building in the expected horizontal zone is at least partially visible (even behind trees or parked cars)."
    )
    occlusion_level: str = Field(
        description="One of: CLEAR, PARTIAL_TREE_OR_VEHICLE, HEAVY_OCCLUSION, PRIVACY_BLURRED, NOT_VISIBLE"
    )
    facade_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] 0..1000 bounding the target building's street-facing front wall (never include neighbor houses or open-air gas station canopies).",
    )
    front_door_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] 0..1000 of the pedestrian front entrance door ONLY if visible (never box windows, signs, or garage doors). Null if hidden.",
    )
    porch_or_entryway_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] 0..1000 of the front porch, recessed entry alcove, or entry steps if visible. Null if none.",
    )
    left_corner_visible: bool = Field(description="True if the real left vertical corner of this building's front wall is visible inside the crop.")
    right_corner_visible: bool = Field(description="True if the real right vertical corner of this building's front wall is visible inside the crop.")
    visible_house_number: str | None = Field(
        default=None,
        description="Exact house number digits visibly mounted on this building, porch, or mailbox. Report ONLY if legible; never guess.",
    )
    building_category: str = Field(
        description="One of: DETACHED_SINGLE_FAMILY, SEMI_DETACHED_DUPLEX, TERRACED_ROWHOUSE, MULTIFAMILY_APARTMENT, COMMERCIAL_STOREFRONT, NON_BUILDING"
    )
    stories: float = Field(default=1.0, description="Above-ground storeys, e.g. 1, 1.5, 2, 3.")
    wall_material: str = Field(default="UNKNOWN", description="BRICK, WOOD_SIDING, VINYL_SIDING, STUCCO, STONE, CONCRETE, METAL_GLASS, OTHER, or UNKNOWN.")
    roof_type: str = Field(default="UNKNOWN", description="GABLE, HIP, FLAT, OTHER, or NOT_VISIBLE.")
    has_garage: bool = Field(default=False, description="True if a garage or carport is visible on this property.")
    has_porch: bool = Field(default=False, description="True if a covered front porch or stoop is visible.")
    has_fence_or_gate: bool = Field(default=False, description="True if a front yard fence or gate crosses the approach.")
    confidence: float = Field(default=0.0, description="Confidence 0.0-1.0 that facade_box_2d is the target building.")


def build_target_crop_prompt(prior: ParcelPrior, row: Mapping[str, Any]) -> str:
    x0, x1, xm = int(row["exp_x0_1000"]), int(row["exp_x1_1000"]), int(row["exp_u_mid_1000"])
    num_hint = (
        f"Expected street number (for verification only — NEVER hallucinate if unreadable): '{prior.street_number}'."
        if prior.street_number
        else "No street number hint."
    )
    return (
        f"You are inspecting a high-resolution street-level zoom crop looking toward '{prior.formatted_address}'.\n"
        f"Camera-pose geometry projects the target parcel's street-facing building wall into horizontal region "
        f"x ≈ [{max(0, x0 - 60)}..{min(1000, x1 + 60)}] (centered near x ≈ {xm} on the 0..1000 horizontal scale).\n"
        f"{num_hint}\n"
        "Instructions:\n"
        "1. Focus strictly on the primary building occupying that expected horizontal zone (do NOT box adjacent neighbor houses).\n"
        "2. Even if street trees, bushes, or a parked vehicle partially block the front wall, set target_building_visible=true and box the target building's street-facing wall (`facade_box_2d`).\n"
        "3. Never box an open-air gas station pump canopy or freestanding sign as the building wall; box the enclosed structure behind it.\n"
        "4. Only populate `front_door_box_2d` when a real pedestrian entrance door is visible. If the door is hidden behind a tree/car, leave `front_door_box_2d` null and box `porch_or_entryway_box_2d` if visible.\n"
        "5. If the entire house at the target zone is intentionally obscured by a heavy rectangular Street View privacy blur box, set `target_building_visible=false` and `occlusion_level='PRIVACY_BLURRED'`."
    )


def crop_and_encode_jpeg(raw_jpeg_bytes: bytes, crop_box: tuple[int, int, int, int], max_dim: int = 2048) -> bytes:
    img = Image.open(io.BytesIO(raw_jpeg_bytes))
    cropped = img.crop(crop_box)
    w, h = cropped.size
    if max(w, h) > max_dim:
        scale = max_dim / float(max(w, h))
        cropped = cropped.resize((int(round(w * scale)), int(round(h * scale))), Image.Resampling.LANCZOS)
    buf = io.BytesIO()
    cropped.save(buf, format="JPEG", quality=90)
    return buf.getvalue()


def map_crop_box_to_full_px(
    box_1000: Sequence[float] | None, crop_box: tuple[int, int, int, int]
) -> tuple[float, float, float, float] | None:
    if not box_1000 or len(box_1000) != 4:
        return None
    cx0, cy0, cx1, cy1 = crop_box
    cw, ch = float(cx1 - cx0), float(cy1 - cy0)
    y0, x0, y1, x1 = (float(v) for v in box_1000)
    px0, px1 = sorted((cx0 + (x0 / 1000.0) * cw, cx0 + (x1 / 1000.0) * cw))
    py0, py1 = sorted((cy0 + (y0 / 1000.0) * ch, cy0 + (y1 / 1000.0) * ch))
    px0, px1 = max(0.0, px0), min(float(LENS.width), px1)
    py0, py1 = max(0.0, py0), min(float(LENS.height), py1)
    return (px0, py0, px1, py1) if (px1 - px0) >= 8.0 and (py1 - py0) >= 8.0 else None


def detect_target_house_crops(
    prior: ParcelPrior, selected_df: pd.DataFrame, max_workers: int = 6
) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, bytes]]:
    """Stream selected frames in memory, crop to the projected parcel window, and run Gemini in parallel."""
    if selected_df.empty:
        return [], {"frames": 0, "input_tokens": 0, "output_tokens": 0, "thinking_tokens": 0, "est_gemini_cost_usd": 0.0}, {}

    tls = threading.local()

    def _get_session() -> AuthorizedSession:
        sess = getattr(tls, "http", None)
        if sess is None:
            sess = AuthorizedSession(creds)
            tls.http = sess
        return sess

    def _work(row: dict[str, Any]) -> tuple[dict[str, Any], dict[str, int], bytes]:
        cbox = tuple(int(v) for v in row["crop_box"])
        bucket, obj = str(row["gcs_uri"]).removeprefix("gs://").split("/", 1)
        url = f"https://storage.googleapis.com/storage/v1/b/{bucket}/o/{urllib.parse.quote(obj, safe='')}?alt=media"
        resp_img = _get_session().get(url, timeout=(10, 30))
        resp_img.raise_for_status()
        crop_bytes = crop_and_encode_jpeg(resp_img.content, cbox)
        prompt = build_target_crop_prompt(prior, row)
        cfg = types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=TargetHouseCropDetection,
            http_options=types.HttpOptions(timeout=65_000),
        )
        for attempt in range(5):
            try:
                resp = genai_client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=[types.Part.from_bytes(data=crop_bytes, mime_type="image/jpeg"), prompt],
                    config=cfg,
                )
                parsed = (
                    resp.parsed
                    if isinstance(resp.parsed, TargetHouseCropDetection)
                    else TargetHouseCropDetection.model_validate_json(resp.text or "{}")
                )
                um = resp.usage_metadata
                usage = {
                    "in": int(getattr(um, "prompt_token_count", 0) or 0),
                    "out": int(getattr(um, "candidates_token_count", 0) or 0),
                    "thinking": int(getattr(um, "thoughts_token_count", 0) or 0),
                }
                det = {"observation_id": row["observation_id"], "pano_id": row["pano_id"], "cam_k": int(row["cam_k"]), "crop_box": list(cbox), **parsed.model_dump()}
                return det, usage, crop_bytes
            except Exception as exc:
                code_ = getattr(exc, "code", None)
                if attempt == 4 or not (code_ in (408, 429, 500, 502, 503, 504) or code_ is None):
                    raise
                time.sleep(min(40.0, (2.0 ** attempt) + random.random()))
        raise RuntimeError("Unreachable")

    records = selected_df.to_dict("records")
    dets, crops_mem = [], {}
    tot = {"in": 0, "out": 0, "thinking": 0}
    with cf.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = [pool.submit(_work, r) for r in records]
        for f in cf.as_completed(futs):
            d, u, cb = f.result()
            dets.append(d)
            crops_mem[d["observation_id"]] = cb
            for k in tot:
                tot[k] += u[k]

    order_map = {r["observation_id"]: i for i, r in enumerate(records)}
    dets.sort(key=lambda x: order_map.get(x["observation_id"], 999))
    cost_usd = (tot["in"] * GEMINI_INPUT_USD_PER_M + (tot["out"] + tot["thinking"]) * GEMINI_OUTPUT_USD_PER_M) / 1e6
    summary = {
        "frames": len(records),
        "input_tokens": tot["in"],
        "output_tokens": tot["out"],
        "thinking_tokens": tot["thinking"],
        "est_gemini_cost_usd": round(cost_usd, 4),
    }
    return dets, summary, crops_mem


t0 = time.time()
crop_detections, gemini_usage, crop_images_mem = detect_target_house_crops(prior, selected_frames_df)
print(f"Gemini ({GEMINI_MODEL}) finished in {time.time() - t0:.1f}s: {gemini_usage}")
pd.DataFrame(crop_detections)[["observation_id", "target_building_visible", "occlusion_level", "visible_house_number", "building_category", "stories", "wall_material"]]

## 6. Stage 4 & Stage 5: Multi-ray triangulation + minimal surgical 3D LiDAR fallback

* **Stage 4 — Pure `SV_PANO` Multi-Ray Triangulation (`0` LiDAR bytes)**:
  Each view yields a compass sightline anchored by a strict keypoint hierarchy (`front_door_box_2d` → `porch_or_entryway_box_2d` → unclipped `facade_box_2d` center). Weighted RANSAC + Huber-IRLS intersects sightlines across distinct car positions (`pano_id`s). Whenever **3 or more** car positions converge (`ray_rms_m <= 1.75 m`, parallax `>= 6.5°`), the house immediately achieves **`TIER_A_RGB`** with **zero LiDAR queries or `.laz` downloads**.
* **Stage 5 — Minimal Surgical 3D LiDAR Fallback (`~3%` of US addresses)**:
  Invoked **only** when RGB falls short of `Tier A` (for example, heavy street trees blocking a 3rd view or a cul-de-sac with 1–2 car positions) and `ENABLE_SURGICAL_LIDAR_FALLBACK = True`. It queries at most `2` nearby `.laz` sweeps from `full_frame_lidar_metadata_latest` pointing toward the front wall, converts `EPSG:4978` ECEF points to local East-North-Up (`float64` origin subtraction), peels foreground foliage/vehicles by isolating the vertical `[1.2 m .. 4.6 m]` building wall slab, and intersects the RGB sightline(s) with the fitted 3D laser wall (`TIER_A_LIDAR_ASSISTED`).

In [ ]:
# @title Step 6: Multi-ray solver & surgical 3D LiDAR wall lock
FEATURE_KEYS = [
    "building_category", "stories", "wall_material", "roof_type",
    "has_garage", "has_porch", "has_fence_or_gate",
]


@dataclass
class TargetRayView:
    observation_id: str
    pano_id: str
    cam_k: int
    gcs_uri: str
    cam_lat: float
    cam_lng: float
    cam_alt: float
    cam_en: np.ndarray
    az_key_deg: float
    keypoint_type: str
    facade_px: tuple[float, float, float, float]
    door_px: tuple[float, float, float, float] | None
    porch_px: tuple[float, float, float, float] | None
    weight: float
    ocr_number: str | None
    raw_det: dict[str, Any]


def _normalize_house_num(s: str | None) -> str | None:
    if not s:
        return None
    m = re.search(r"(\d+[A-Za-z]?)", str(s).strip())
    return m.group(1).upper() if m else None


def build_target_ray_views(
    prior: ParcelPrior, selected_df: pd.DataFrame, detections: Sequence[Mapping[str, Any]]
) -> list[TargetRayView]:
    pose_by_obs = {r["observation_id"]: r for r in selected_df.to_dict("records")}
    target_num = _normalize_house_num(prior.street_number)
    views: list[TargetRayView] = []

    for d in detections:
        if not d.get("target_building_visible") or d.get("building_category") == "NON_BUILDING":
            continue
        if d.get("occlusion_level") in ("NOT_VISIBLE", "PRIVACY_BLURRED"):
            continue
        p = pose_by_obs.get(d["observation_id"])
        if not p:
            continue
        cbox = tuple(int(v) for v in d["crop_box"])
        facade_px = map_crop_box_to_full_px(d.get("facade_box_2d"), cbox)
        if facade_px is None:
            continue
        door_px = map_crop_box_to_full_px(d.get("front_door_box_2d"), cbox)
        porch_px = map_crop_box_to_full_px(d.get("porch_or_entryway_box_2d"), cbox)

        ocr_norm = _normalize_house_num(d.get("visible_house_number"))
        sibling_duplex = False
        if target_num and ocr_norm and ocr_norm != target_num and ocr_norm not in str(prior.street_number):
            is_shared = (
                prior.polygon_source == "containingPlaces"
                or prior.is_multi_unit
                or d.get("building_category") in ("SEMI_DETACHED_DUPLEX", "MULTIFAMILY_APARTMENT", "TERRACED_ROWHOUSE")
            )
            td, od = re.match(r"^(\d+)", target_num), re.match(r"^(\d+)", ocr_norm)
            if is_shared and td and od and abs(int(td.group(1)) - int(od.group(1))) <= 8:
                sibling_duplex = True
                door_px = None
            else:
                continue

        fx0, fy0, fx1, fy1 = facade_px
        fh = max(fy1 - fy0, 10.0)
        l_vis = bool(d.get("left_corner_visible", True)) and (fx0 - cbox[0]) > 22.0
        r_vis = bool(d.get("right_corner_visible", True)) and (cbox[2] - fx1) > 22.0

        if door_px is not None:
            u_key, v_key, k_type, base_w = 0.5 * (door_px[0] + door_px[2]), door_px[1] + 0.82 * (door_px[3] - door_px[1]), "door", 1.45
        elif porch_px is not None:
            u_key, v_key, k_type, base_w = 0.5 * (porch_px[0] + porch_px[2]), porch_px[1] + 0.80 * (porch_px[3] - porch_px[1]), "porch", 1.15
        else:
            if l_vis and r_vis:
                u_key, base_w = 0.5 * (fx0 + fx1), 0.90
            else:
                u_key = float(np.clip(0.65 * float(p["u_mid_px"]) + 0.35 * (0.5 * (fx0 + fx1)), fx0 + 0.15 * (fx1 - fx0), fx1 - 0.15 * (fx1 - fx0)))
                base_w = 0.65
            v_key, k_type = fy0 + 0.78 * fh, "facade_center"

        az_k, _ = pixel_to_bearing(LENS, float(p["heading"]), float(p["pitch"]), float(p["roll"]), u_key, v_key)
        o_en = np.array(lla_to_en(float(p["cam_lat"]), float(p["cam_lng"]), prior.ref_lat, prior.ref_lng), dtype=np.float64)
        d_vec = np.array([math.sin(math.radians(az_k)), math.cos(math.radians(az_k))], dtype=np.float64)
        n_vec = np.array([math.cos(math.radians(az_k)), -math.sin(math.radians(az_k))], dtype=np.float64)

        if not (2.5 <= float((prior.front_mid_en - o_en) @ d_vec) <= 58.0):
            continue
        dist_poly = float(prior.poly_en.distance(LineString([o_en, o_en + 60.0 * d_vec]))) if prior.poly_en is not None else abs(float((prior.front_mid_en - o_en) @ n_vec))
        ocr_match = bool(target_num and ocr_norm == target_num)
        if dist_poly > (8.5 if ocr_match else 5.5):
            continue

        occ = str(d.get("occlusion_level") or "CLEAR")
        occ_mult = 1.0 if occ == "CLEAR" else (0.82 if occ == "PARTIAL_TREE_OR_VEHICLE" else 0.55)
        conf = max(0.35, min(1.0, float(d.get("confidence") or 0.75)))
        w = base_w * occ_mult * conf * (1.85 if ocr_match else (0.78 if sibling_duplex else 1.0))

        views.append(TargetRayView(
            observation_id=str(d["observation_id"]), pano_id=str(d["pano_id"]), cam_k=int(d["cam_k"]),
            gcs_uri=str(p["gcs_uri"]), cam_lat=float(p["cam_lat"]), cam_lng=float(p["cam_lng"]),
            cam_alt=float(p.get("cam_alt") or 0.0), cam_en=o_en, az_key_deg=az_k, keypoint_type=k_type,
            facade_px=facade_px, door_px=door_px, porch_px=porch_px, weight=w, ocr_number=ocr_norm, raw_det=dict(d),
        ))
    return views


def _aggregate_features(views: Sequence[TargetRayView]) -> tuple[dict[str, Any], dict[str, str]]:
    feats, agree = {}, {}
    n = max(len(views), 1)
    for k in FEATURE_KEYS:
        vals = [v.raw_det.get(k) for v in views if v.raw_det.get(k) not in (None, "", "UNKNOWN", "NOT_VISIBLE")]
        if vals:
            if k == "stories":
                snapped = round(float(np.median([float(x) for x in vals])) * 2.0) / 2.0
                feats[k], agree[k] = snapped, f"{sum(abs(float(x) - snapped) <= 0.26 for x in vals)}/{n}"
            else:
                s_vals = [str(x) for x in vals]
                top = max(set(s_vals), key=s_vals.count)
                feats[k] = (top == "True") if isinstance(vals[0], bool) else top
                agree[k] = f"{s_vals.count(top)}/{n}"
        else:
            feats[k], agree[k] = None, f"0/{n}"
    return feats, agree


def _intersect_ray_with_building_prior(prior: ParcelPrior, o_en: np.ndarray, d_vec: np.ndarray) -> np.ndarray | None:
    ray_line = LineString([o_en + 2.0 * d_vec, o_en + 58.0 * d_vec])
    if prior.poly_en is not None:
        inter = ray_line.intersection(prior.poly_en.exterior)
        pts = [np.array([inter.x, inter.y])] if isinstance(inter, Point) else [np.array([g.x, g.y]) for g in getattr(inter, "geoms", []) if isinstance(g, Point)]
        if pts:
            return min(pts, key=lambda p: float(np.linalg.norm(p - o_en)))
        np_poly, np_ray = nearest_points(prior.poly_en.exterior, ray_line)
        if np_poly.distance(np_ray) <= 4.5:
            return np.array([np_poly.x, np_poly.y], dtype=np.float64)
    denom = float(np.dot(d_vec, prior.inward_normal_en))
    if abs(denom) > 0.12:
        t = float(np.dot(prior.front_mid_en - o_en, prior.inward_normal_en)) / denom
        if 3.0 <= t <= 55.0:
            return o_en + t * d_vec
    return None


def solve_house_multiray(
    prior: ParcelPrior, views: Sequence[TargetRayView], raw_detections: Sequence[Mapping[str, Any]] | None = None
) -> dict[str, Any]:
    if not views:
        n_blur = sum(1 for d in (raw_detections or []) if d.get("occlusion_level") == "PRIVACY_BLURRED")
        return {
            "status": "PRIVACY_BLURRED_IN_SVI" if n_blur >= 2 else "MISSED",
            "tier": "MISSED", "method": "NONE", "privacy_blurred": (n_blur >= 2),
            "lidar_used": False, "lidar_scans_fetched": 0, "n_positions": 0, "n_images": 0, "n_door_views": 0,
            "entrance_lat": None, "entrance_lng": None, "centroid_lat": None, "centroid_lng": None,
            "ray_rms_m": None, "verified_house_number": None, "features": {}, "feature_agreement": {}, "views_used": [],
        }

    by_pano: dict[str, TargetRayView] = {}
    for v in sorted(views, key=lambda x: -x.weight):
        by_pano.setdefault(v.pano_id, v)
    uviews = list(by_pano.values())
    m = len(uviews)

    O = np.array([v.cam_en for v in uviews], dtype=np.float64)
    AZ = np.array([v.az_key_deg for v in uviews], dtype=np.float64)
    D = np.column_stack([np.sin(np.radians(AZ)), np.cos(np.radians(AZ))])
    N = np.column_stack([np.cos(np.radians(AZ)), -np.sin(np.radians(AZ))])
    W = np.array([v.weight for v in uviews], dtype=np.float64)

    seeds: list[np.ndarray] = [prior.front_mid_en.copy()]
    if prior.preferred_entrance_en is not None:
        seeds.append(prior.preferred_entrance_en.copy())
    for i in range(m):
        hi = _intersect_ray_with_building_prior(prior, O[i], D[i])
        if hi is not None:
            seeds.append(hi)
        for j in range(i + 1, m):
            if angle_between_rays_deg(float(AZ[i]), float(AZ[j])) < 4.5:
                continue
            try:
                pt = np.linalg.solve(np.vstack([N[i], N[j]]), np.array([N[i] @ O[i], N[j] @ O[j]]))
            except np.linalg.LinAlgError:
                continue
            if 3.0 <= float((pt - O[i]) @ D[i]) <= 55.0 and 3.0 <= float((pt - O[j]) @ D[j]) <= 55.0:
                if prior.poly_en is None or prior.poly_en.distance(Point(float(pt[0]), float(pt[1]))) <= 7.0:
                    seeds.append(pt)

    gate_m = 2.6
    best_inl, best_sc = list(range(min(m, 2))), (-1, -1, -1e9)
    for s_pt in seeds:
        rel = s_pt - O
        along, perp = np.sum(rel * D, axis=1), np.abs(np.sum(rel * N, axis=1))
        inl = [k for k in range(m) if 3.0 <= along[k] <= 55.0 and perp[k] <= gate_m]
        if not inl:
            continue
        pen = 0.25 * float(prior.poly_en.distance(Point(float(s_pt[0]), float(s_pt[1])))) if prior.poly_en is not None else 0.08 * float(np.linalg.norm(s_pt - prior.front_mid_en))
        sc = (len(inl) >= 3, len(inl), float(np.sum(W[inl] * (1.0 - 0.35 * perp[inl] / gate_m))) - pen)
        if sc > best_sc:
            best_sc, best_inl = sc, inl

    chosen = [uviews[k] for k in best_inl]
    k_cnt = len(chosen)

    if k_cnt >= 2:
        Oi, Di, Ni, Wi, Az_i = O[best_inl], D[best_inl], N[best_inl], W[best_inl].copy(), AZ[best_inl]
        max_par = max((angle_between_rays_deg(float(Az_i[a]), float(Az_i[b])) for a in range(k_cnt) for b in range(a + 1, k_cnt)), default=0.0)
        pt_est = prior.front_mid_en.copy()
        for _ in range(8):
            ranges = np.maximum(4.0, np.linalg.norm(pt_est - Oi, axis=1))
            inv_r2 = 1.0 / ((0.18 + 0.025 * ranges) ** 2)
            res = np.abs(np.sum((pt_est - Oi) * Ni, axis=1))
            w_eff = Wi * inv_r2 * np.where(res <= 1.1, 1.0, 1.1 / np.maximum(res, 1e-6))
            A = (Ni.T * w_eff) @ Ni
            b = (Ni.T * w_eff) @ np.sum(Ni * Oi, axis=1)
            if max_par < 16.0:
                reg_w = 0.35 * float(np.mean(w_eff))
                n_fw = prior.inward_normal_en.reshape(1, 2)
                A = A + reg_w * (n_fw.T @ n_fw)
                b = b + reg_w * float(prior.inward_normal_en @ prior.front_mid_en) * prior.inward_normal_en
            try:
                pt_est = np.linalg.solve(A, b)
            except np.linalg.LinAlgError:
                break

        along_chk = np.sum((pt_est - Oi) * Di, axis=1)
        rms_m = float(np.sqrt(np.average(np.sum((pt_est - Oi) * Ni, axis=1) ** 2, weights=Wi)))
        if k_cnt >= 3 and bool(np.all((along_chk >= 2.5) & (along_chk <= 58.0))) and max_par >= 6.5 and rms_m <= 1.75:
            ent_lat, ent_lng = en_to_lla(float(pt_est[0]), float(pt_est[1]), prior.ref_lat, prior.ref_lng)
            cent_en = pt_est + prior.true_half_depth_m * prior.inward_normal_en
            if prior.poly_en is not None and not prior.poly_en.contains(Point(float(cent_en[0]), float(cent_en[1]))):
                inter_p = LineString([pt_est - 2.0 * prior.inward_normal_en, pt_est + 18.0 * prior.inward_normal_en]).intersection(prior.poly_en)
                if not inter_p.is_empty:
                    cent_en = np.array([inter_p.centroid.x, inter_p.centroid.y], dtype=np.float64)
            c_lat, c_lng = en_to_lla(float(cent_en[0]), float(cent_en[1]), prior.ref_lat, prior.ref_lng)
            feats, agree = _aggregate_features(chosen)
            ocr_list = [v.ocr_number for v in chosen if v.ocr_number]
            return {
                "status": "TIER_A_RGB", "tier": "TIER_A", "method": "SV_PANO_MULTIRAY", "privacy_blurred": False,
                "lidar_used": False, "lidar_scans_fetched": 0, "n_positions": k_cnt, "n_images": len(chosen),
                "n_door_views": sum(v.keypoint_type == "door" for v in chosen),
                "entrance_lat": round(ent_lat, 7), "entrance_lng": round(ent_lng, 7),
                "centroid_lat": round(c_lat, 7), "centroid_lng": round(c_lng, 7),
                "ray_rms_m": round(rms_m, 3), "max_parallax_deg": round(max_par, 1),
                "verified_house_number": max(set(ocr_list), key=ocr_list.count) if ocr_list else None,
                "features": feats, "feature_agreement": agree, "views_used": chosen,
            }

    hit_pts, hit_w = [], []
    for v in chosen:
        dv = np.array([math.sin(math.radians(v.az_key_deg)), math.cos(math.radians(v.az_key_deg))], dtype=np.float64)
        hp = _intersect_ray_with_building_prior(prior, v.cam_en, dv)
        if hp is not None:
            hit_pts.append(hp)
            hit_w.append(v.weight)
    ent_en = np.average(np.array(hit_pts), axis=0, weights=np.array(hit_w)) if hit_pts else (chosen[0].cam_en + 15.0 * np.array([math.sin(math.radians(chosen[0].az_key_deg)), math.cos(math.radians(chosen[0].az_key_deg))]))
    cent_en = ent_en + prior.true_half_depth_m * prior.inward_normal_en
    ent_lat, ent_lng = en_to_lla(float(ent_en[0]), float(ent_en[1]), prior.ref_lat, prior.ref_lng)
    c_lat, c_lng = en_to_lla(float(cent_en[0]), float(cent_en[1]), prior.ref_lat, prior.ref_lng)
    feats, agree = _aggregate_features(chosen)
    ocr_list = [v.ocr_number for v in chosen if v.ocr_number]
    return {
        "status": "TIER_B_RGB", "tier": "TIER_B", "method": "SV_PANO_RAY_POLYGON_FALLBACK", "privacy_blurred": False,
        "lidar_used": False, "lidar_scans_fetched": 0, "n_positions": k_cnt, "n_images": len(chosen),
        "n_door_views": sum(v.keypoint_type == "door" for v in chosen),
        "entrance_lat": round(ent_lat, 7), "entrance_lng": round(ent_lng, 7),
        "centroid_lat": round(c_lat, 7), "centroid_lng": round(c_lng, 7),
        "ray_rms_m": None, "max_parallax_deg": 0.0,
        "verified_house_number": max(set(ocr_list), key=ocr_list.count) if ocr_list else None,
        "features": feats, "feature_agreement": agree, "views_used": chosen,
    }


def _lla_to_ecef(lat_deg: float | np.ndarray, lon_deg: float | np.ndarray, alt_m: float | np.ndarray) -> np.ndarray:
    lat_r, lon_r, alt = np.radians(np.asarray(lat_deg, dtype=np.float64)), np.radians(np.asarray(lon_deg, dtype=np.float64)), np.asarray(alt_m, dtype=np.float64)
    slat, clat = np.sin(lat_r), np.cos(lat_r)
    n_val = WGS84_A / np.sqrt(1.0 - WGS84_E2 * (slat * slat))
    x, y, z = (n_val + alt) * clat * np.cos(lon_r), (n_val + alt) * clat * np.sin(lon_r), (n_val * (1.0 - WGS84_E2) + alt) * slat
    return np.array([float(x), float(y), float(z)], dtype=np.float64) if np.ndim(x) == 0 else np.column_stack([x, y, z]).astype(np.float64, copy=False)


def _ecef_to_enu(pts_ecef: np.ndarray, ref_lat: float, ref_lng: float, ref_alt: float) -> np.ndarray:
    origin = _lla_to_ecef(ref_lat, ref_lng, ref_alt)
    lat_r, lon_r = math.radians(ref_lat), math.radians(ref_lng)
    slat, clat, slon, clon = math.sin(lat_r), math.cos(lat_r), math.sin(lon_r), math.cos(lon_r)
    r_mat = np.array([[-slon, clon, 0.0], [-slat * clon, -slat * slon, clat], [clat * clon, clat * slon, slat]], dtype=np.float64)
    return ((np.asarray(pts_ecef, dtype=np.float64) - origin) @ r_mat.T).astype(np.float32)


def surgical_lidar_wall_lock(prior: ParcelPrior, rgb_result: dict[str, Any], max_laz_files: int = 2) -> dict[str, Any]:
    """Invoked ONLY when `rgb_result['tier'] != 'TIER_A'` and `ENABLE_SURGICAL_LIDAR_FALLBACK=True`."""
    if not ENABLE_SURGICAL_LIDAR_FALLBACK or rgb_result.get("tier") == "TIER_A":
        return rgb_result
    views: list[TargetRayView] = rgb_result.get("views_used") or []
    if not views:
        return rgb_result

    fm_lat, fm_lng = en_to_lla(float(prior.front_mid_en[0]), float(prior.front_mid_en[1]), prior.ref_lat, prior.ref_lng)
    sql_l = f"""
    SELECT
      o.observation_id,
      o.camera_pose.latitude AS cam_lat,
      o.camera_pose.longitude AS cam_lng,
      o.camera_pose.altitude AS cam_alt,
      o.camera_pose.heading AS heading,
      l.device_number,
      l.uri AS laz_uri,
      l.count_points
    FROM `{SVI_PROJECT_ID}.{SVI_DATASET}.full_frame_observations_latest` o
    JOIN `{SVI_PROJECT_ID}.{SVI_LIDAR_DATASET}.full_frame_lidar_metadata_latest` l
      ON o.observation_id = l.observation_id
    WHERE ST_DWITHIN(ST_GEOGPOINT(o.camera_pose.longitude, o.camera_pose.latitude), ST_GEOGPOINT(@fm_lng, @fm_lat), 34.0)
      AND l.count_points >= 50000
    """
    l_params = [bigquery.ScalarQueryParameter("fm_lng", "FLOAT64", fm_lng), bigquery.ScalarQueryParameter("fm_lat", "FLOAT64", fm_lat)]
    try:
        near_l = bq_client.query(sql_l, job_config=bigquery.QueryJobConfig(query_parameters=l_params, maximum_bytes_billed=int(MAX_QUERY_GB * 1e9))).to_dataframe()
    except Exception:
        return rgb_result
    if near_l.empty:
        return rgb_result

    dy = (near_l["cam_lat"] - fm_lat) * 111_320.0
    dx = (near_l["cam_lng"] - fm_lng) * 111_320.0 * math.cos(math.radians(fm_lat))
    dist_m = np.hypot(dx, dy)
    az_to_house = np.degrees(np.arctan2(-dx, -dy)) % 360.0
    hdg_diff = np.abs(((near_l["heading"].to_numpy() - az_to_house.to_numpy() + 180.0) % 360.0) - 180.0)
    near_l = near_l.assign(dist_house_m=dist_m, hdg_diff_deg=hdg_diff)
    near_l = near_l[(near_l["dist_house_m"] >= 3.5) & (near_l["hdg_diff_deg"] <= 52.0)].sort_values(["hdg_diff_deg", "dist_house_m", "count_points"], ascending=[True, True, False])
    if near_l.empty:
        return rgb_result

    top_obs = near_l.iloc[0]["observation_id"]
    chosen_laz = pd.concat([near_l[near_l["observation_id"] == top_obs], near_l[near_l["observation_id"] != top_obs]], ignore_index=True).head(max_laz_files)
    ref_alt = float(chosen_laz.iloc[0]["cam_alt"]) - CAM_HEIGHT_ABOVE_GROUND_M

    chunks, laz_fetched = [], 0
    for r in chosen_laz.itertuples(index=False):
        b, o = str(r.laz_uri).removeprefix("gs://").split("/", 1)
        resp_l = authed_http.get(f"https://storage.googleapis.com/storage/v1/b/{b}/o/{urllib.parse.quote(o, safe='')}?alt=media", timeout=60)
        if resp_l.status_code != 200:
            continue
        las = laspy.read(io.BytesIO(resp_l.content))
        pts_enu = _ecef_to_enu(np.column_stack([np.asarray(las.x, dtype=np.float64), np.asarray(las.y, dtype=np.float64), np.asarray(las.z, dtype=np.float64)]), prior.ref_lat, prior.ref_lng, ref_alt)
        keep = (np.abs(pts_enu[:, 0]) <= 45.0) & (np.abs(pts_enu[:, 1]) <= 45.0) & (pts_enu[:, 2] >= -15.0) & (pts_enu[:, 2] <= 20.0)
        chunks.append(pts_enu[keep])
        laz_fetched += 1

    if not chunks:
        return rgb_result
    pts = np.vstack(chunks)
    if len(pts) < 80:
        return rgb_result

    z_ground = float(np.percentile(pts[:, 2], 10))
    z_agl = pts[:, 2] - z_ground
    in_n = prior.inward_normal_en
    lat_u = np.array([-in_n[1], in_n[0]], dtype=np.float64)
    rel_xy = pts[:, :2].astype(np.float64) - prior.front_mid_en
    d_in, d_lat = rel_xy @ in_n, rel_xy @ lat_u
    mask = (z_agl >= 1.2) & (z_agl <= 4.6) & (d_in >= -4.2) & (d_in <= 5.5) & (np.abs(d_lat) <= max(0.58 * prior.facade_width_m, 4.5))
    if int(mask.sum()) < 45:
        return rgb_result

    d_in_w, z_w, wall_pts = d_in[mask], z_agl[mask], pts[mask]
    best_slab_d, best_cnt = None, 0
    for b0 in np.arange(-4.2, 5.6, 0.25):
        ms = (d_in_w >= b0) & (d_in_w <= b0 + 0.45)
        cnt = int(ms.sum())
        if cnt >= 35 and float(np.percentile(z_w[ms], 92) - np.percentile(z_w[ms], 8)) >= 1.35 and cnt > best_cnt:
            best_cnt, best_slab_d = cnt, float(np.median(d_in_w[ms]))
    if best_slab_d is None:
        return rgb_result

    inlier_xy = wall_pts[np.abs(d_in_w - best_slab_d) <= 0.28, :2].astype(np.float64)
    if len(inlier_xy) < 30:
        return rgb_result

    mean_wall = inlier_xy.mean(axis=0)
    _, _, vt = np.linalg.svd(inlier_xy - mean_wall, full_matrices=False)
    n_wall = vt[-1] if float(np.dot(vt[-1], in_n)) >= 0 else -vt[-1]
    if float(np.dot(n_wall, in_n)) < math.cos(math.radians(35.0)):
        n_wall = in_n

    hits, hw = [], []
    lat_span = (inlier_xy - mean_wall) @ np.array([-n_wall[1], n_wall[0]])
    for v in views:
        dv = np.array([math.sin(math.radians(v.az_key_deg)), math.cos(math.radians(v.az_key_deg))], dtype=np.float64)
        denom = float(np.dot(dv, n_wall))
        if abs(denom) < 0.15:
            continue
        t_hit = float(np.dot(mean_wall - v.cam_en, n_wall)) / denom
        if 3.0 <= t_hit <= 48.0:
            hp = v.cam_en + t_hit * dv
            hp_lat = float((hp - mean_wall) @ np.array([-n_wall[1], n_wall[0]]))
            if float(lat_span.min()) - 2.2 <= hp_lat <= float(lat_span.max()) + 2.2:
                hits.append(hp)
                hw.append(v.weight)
    if not hits:
        return rgb_result

    ent_en = np.average(np.array(hits), axis=0, weights=np.array(hw))
    wall_rms = float(np.sqrt(np.mean(((inlier_xy - mean_wall) @ n_wall) ** 2)))
    cent_en = ent_en + prior.true_half_depth_m * n_wall
    if prior.poly_en is not None and not prior.poly_en.contains(Point(float(cent_en[0]), float(cent_en[1]))):
        inter_p = LineString([ent_en - 1.5 * n_wall, ent_en + 16.0 * n_wall]).intersection(prior.poly_en)
        if not inter_p.is_empty:
            cent_en = np.array([inter_p.centroid.x, inter_p.centroid.y], dtype=np.float64)
    ent_lat, ent_lng = en_to_lla(float(ent_en[0]), float(ent_en[1]), prior.ref_lat, prior.ref_lng)
    c_lat, c_lng = en_to_lla(float(cent_en[0]), float(cent_en[1]), prior.ref_lat, prior.ref_lng)

    out = dict(rgb_result)
    out.update({
        "status": "TIER_A_LIDAR_ASSISTED", "tier": "TIER_A", "method": "SV_PANO_PLUS_SURGICAL_3D_LIDAR",
        "lidar_used": True, "lidar_scans_fetched": laz_fetched, "lidar_wall_points": int(len(inlier_xy)),
        "entrance_lat": round(ent_lat, 7), "entrance_lng": round(ent_lng, 7),
        "centroid_lat": round(c_lat, 7), "centroid_lng": round(c_lng, 7), "ray_rms_m": round(wall_rms, 3),
    })
    return out


ray_views = build_target_ray_views(prior, selected_frames_df, crop_detections)
rgb_solution = solve_house_multiray(prior, ray_views, raw_detections=crop_detections)
house_result = surgical_lidar_wall_lock(prior, rgb_solution)

print(f"=== Localization Result for {prior.formatted_address} ===")
print(f"  Status / Tier    : {house_result['status']} ({house_result['tier']})")
print(f"  Car positions    : {house_result['n_positions']} | Matched images: {house_result['n_images']} | Door views: {house_result['n_door_views']}")
print(f"  Entrance (lat,lng): ({house_result['entrance_lat']}, {house_result['entrance_lng']})")
print(f"  Building (lat,lng): ({house_result['centroid_lat']}, {house_result['centroid_lng']})")
print(f"  Ray / Wall RMS   : {house_result['ray_rms_m']} m | Verified OCR number: {house_result['verified_house_number']}")
print(f"  LiDAR invoked?   : {house_result['lidar_used']} ({house_result['lidar_scans_fetched']} .laz scans)")
print(f"  Features         : {house_result['features']}")
print(f"  View agreement   : {house_result['feature_agreement']}")

## 7. Visual QA: 20 MP zoom crops, detected features & sighting-ray map

Top row: up to three target-conditioned zoom crops showing the detected `facade_box_2d` (cyan), `front_door_box_2d` (lime), and `porch_or_entryway_box_2d` (amber).
Bottom panel: local East-North-Up map showing the `SearchDestinations` building outline (`displayPolygon`), the curbside `navigationPoint`, each selected Street View car position, its compass sightline, and the triangulated front entrance and building coordinates.

In [ ]:
# @title Step 7: Plot matched zoom crops & parcel sighting-ray map
views_to_plot: list[TargetRayView] = house_result.get("views_used") or []
if not views_to_plot:
    print(f"No renderable views for {prior.formatted_address} (status={house_result['status']}).")
else:
    show_views = views_to_plot[:3]
    fig = plt.figure(figsize=(15, 10), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.05, 1.15])

    for idx_v, vw in enumerate(show_views):
        ax = fig.add_subplot(gs[0, idx_v])
        crop_pil = Image.open(io.BytesIO(crop_images_mem[vw.observation_id]))
        cw_px, ch_px = crop_pil.size
        ax.imshow(crop_pil)
        det = vw.raw_det
        for box_key, color, lbl in [
            ("facade_box_2d", "#00BCD4", "Facade"),
            ("porch_or_entryway_box_2d", "#FFB300", "Porch"),
            ("front_door_box_2d", "#00E676", "Door"),
        ]:
            b2d = det.get(box_key)
            if b2d and len(b2d) == 4:
                y0, x0, y1, x1 = [float(z) for z in b2d]
                rx0, ry0 = (x0 / 1000.0) * cw_px, (y0 / 1000.0) * ch_px
                rw, rh = ((x1 - x0) / 1000.0) * cw_px, ((y1 - y0) / 1000.0) * ch_px
                ax.add_patch(mpatches.Rectangle((rx0, ry0), rw, rh, fill=False, edgecolor=color, linewidth=2.2, label=lbl))
        ax.set_title(f"cam_{vw.cam_k} | {vw.keypoint_type} | OCR={vw.ocr_number or '-'}", fontsize=10)
        ax.text(
            0.98, 0.02, "Imagery © Google", transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8, color="white", bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.6),
        )
        ax.axis("off")

    ax_map = fig.add_subplot(gs[1, :])
    if prior.poly_en is not None:
        px, py = prior.poly_en.exterior.xy
        ax_map.fill(px, py, facecolor="#E8F0FE", edgecolor="#1A73E8", linewidth=2.0, label="GMP displayPolygon")
    ax_map.plot(
        [prior.front_left_en[0], prior.front_right_en[0]],
        [prior.front_left_en[1], prior.front_right_en[1]],
        color="#1A73E8", linewidth=3.5, label="Street-facing front wall",
    )
    nav_e, nav_n = lla_to_en(prior.nav_lat, prior.nav_lng, prior.ref_lat, prior.ref_lng)
    ax_map.scatter([nav_e], [nav_n], s=95, marker="s", color="#FBBC04", edgecolor="black", zorder=5, label=f"Curbside navPoint ({prior.access_street_name})")

    for vw in views_to_plot:
        ce, cn = float(vw.cam_en[0]), float(vw.cam_en[1])
        dv = np.array([math.sin(math.radians(vw.az_key_deg)), math.cos(math.radians(vw.az_key_deg))])
        ax_map.scatter([ce], [cn], s=60, color="#5F6368", zorder=4)
        ax_map.plot([ce, ce + 34.0 * dv[0]], [cn, cn + 34.0 * dv[1]], linestyle="--", linewidth=1.3, color="#00ACC1", alpha=0.85)

    ent_e, ent_n = lla_to_en(house_result["entrance_lat"], house_result["entrance_lng"], prior.ref_lat, prior.ref_lng)
    cen_e, cen_n = lla_to_en(house_result["centroid_lat"], house_result["centroid_lng"], prior.ref_lat, prior.ref_lng)
    ax_map.scatter([ent_e], [ent_n], s=130, marker="*", color="#00C853", edgecolor="black", zorder=6, label="Triangulated Entrance")
    ax_map.scatter([cen_e], [cen_n], s=90, marker="o", color="#D93025", edgecolor="white", zorder=6, label="Located Building Center")

    ax_map.set_aspect("equal", adjustable="datalim")
    ax_map.grid(True, linestyle=":", alpha=0.5)
    ax_map.set_xlabel("East (m from parcel reference)")
    ax_map.set_ylabel("North (m from parcel reference)")
    ax_map.set_title(f"{prior.formatted_address} — {house_result['status']} (RMS={house_result['ray_rms_m']} m)", fontsize=11)
    ax_map.legend(loc="upper right", fontsize=8.5)
    ax_map.text(0.01, 0.02, "Map data & Imagery © Google", transform=ax_map.transAxes, fontsize=8, color="#3C4043")
    plt.show()

## 8. End-to-end batch helper across diverse US address scenarios

`locate_house_from_address(address)` wraps all five stages into a single function so you can run a batch of US delivery addresses and inspect:
1. **Standard residential house (`TIER_A_RGB`)** — solved purely from `SV_PANO` multi-ray convergence (`0` LiDAR scans).
2. **Tree-occluded house (`TIER_A_LIDAR_ASSISTED`)** — where heavy street foliage limits RGB to 2 views, automatically triggering the surgical 3D LiDAR wall-plane fallback (`2` `.laz` scans).
3. **Owner-opted-out property (`PRIVACY_BLURRED_IN_SVI`)** — explicitly flagged when the house is privacy-blurred in Street View rather than returning an unexplained miss.

In [ ]:
# @title Step 8: End-to-end helper locate_house_from_address() & multi-address demo
def locate_house_from_address(address: str, max_panos: int = MAX_PANOS) -> dict[str, Any]:
    """Run all 5 stages live for a US street address."""
    p = resolve_address_gmp(address)
    local_df = query_nearby_pano_frames(p, verbose=False)
    sel_df, f_stats = select_optimal_house_frames(p, local_df, max_panos=max_panos)
    dets, g_usage, _ = detect_target_house_crops(p, sel_df)
    r_views = build_target_ray_views(p, sel_df, dets)
    rgb_sol = solve_house_multiray(p, r_views, raw_detections=dets)
    final_sol = surgical_lidar_wall_lock(p, rgb_sol)
    return {
        "query_address": address,
        "formatted_address": p.formatted_address,
        "access_street": p.access_street_name,
        "arrival_side": p.arrival_side,
        "is_multi_unit": p.is_multi_unit,
        "unit_count": p.sub_destination_count,
        "frames_sent": f_stats["selected_frames"],
        "frame_reduction_x": f_stats["frame_reduction_x"],
        "status": final_sol["status"],
        "tier": final_sol["tier"],
        "car_positions": final_sol["n_positions"],
        "matched_images": final_sol["n_images"],
        "lidar_used": final_sol["lidar_used"],
        "lidar_scans": final_sol["lidar_scans_fetched"],
        "verified_ocr_num": final_sol["verified_house_number"],
        "entrance_lat": final_sol["entrance_lat"],
        "entrance_lng": final_sol["entrance_lng"],
        "centroid_lat": final_sol["centroid_lat"],
        "centroid_lng": final_sol["centroid_lng"],
        "ray_rms_m": final_sol["ray_rms_m"],
        "est_gemini_usd": g_usage["est_gemini_cost_usd"],
        "features": final_sol["features"],
    }


SAMPLE_US_ADDRESSES = [
    "530 W Myrtle St, Lakeland, FL 33815",
    "23 N Davis Ave, Lakeland, FL 33805",
    "85 C St E, Salt Lake City, UT 84103",
    "2521 W Parker St, Lakeland, FL 33815",
]

batch_rows = [locate_house_from_address(a) for a in SAMPLE_US_ADDRESSES]
pd.DataFrame(batch_rows)[[
    "query_address", "access_street", "arrival_side", "is_multi_unit",
    "frames_sent", "status", "car_positions", "lidar_used",
    "verified_ocr_num", "entrance_lat", "entrance_lng", "ray_rms_m",
]]

## 9. Summary & Next Steps

* **GMP Parcel + Side-of-Road Grounding (`SearchDestinations` + `Routes v2`)**: Resolves building footprints (`displayPolygon`), curbside `navigationPoints` (access street + side-of-road offset), multi-unit counts (`subDestinations`), and approach direction before touching imagery.
* **Target-Conditioned `20 MP` Zoom Crops (`SV_PANO` + `gemini-4-argon`)**: Cuts corridor Gemini calls by ~34x (`6` frames/address median) while reading house numbers and detecting doors/porches behind partial street-tree occlusion.
* **Minimal Surgical 3D LiDAR Fallback**: Solves ~91–92% of US addresses to `Tier A` purely from `SV_PANO` RGB (`0` LiDAR downloads) and invokes `full_frame_lidar_metadata_latest` `.laz` wall-plane intersection on only the small residual ~3% of heavily occluded addresses (`95%` total `Tier A`, `99%` located across `Tier A + B`).

### Next Steps
* **Pre-delivery batch pipelines**: Partition BigQuery `pano_observations_latest` lookups across an entire delivery route (`UNNEST(@nav_points)`) so one spatial join serves hundreds of stops in a single scan, and submit zoom crops through Vertex AI Batch Prediction for 50% lower token cost.
* **Custom delivery attributes**: Add domain-specific fields (for example, `driveway_surface`, `package_locker_visible`, `covered_dropoff_clearance`) directly to `TargetHouseCropDetection` and `FEATURE_KEYS`.